# 💰 Expense Tracker App — Exploratory Data Analysis

**Author:** Your Name  
**Project:** Expense Tracker App using Data Science  
**Purpose:** EDA for GitHub portfolio and placement interviews

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data.generate_data import generate_expenses, generate_income

print('✅ Libraries loaded successfully')

In [ ]:
# Generate data
df_exp = generate_expenses()
df_inc = generate_income()

print(f'Expense records: {len(df_exp):,}')
print(f'Income  records: {len(df_inc):,}')
df_exp.head(10)

## 1. Dataset Overview

In [ ]:
print('Shape:', df_exp.shape)
print('\nData Types:')
print(df_exp.dtypes)
print('\nMissing Values:')
print(df_exp.isnull().sum())
print('\nBasic Statistics:')
df_exp['Amount'].describe()

## 2. Category Analysis

In [ ]:
cat_totals = df_exp.groupby('Category')['Amount'].agg(['sum','mean','count'])
cat_totals.columns = ['Total_Spent','Avg_Transaction','Count']
cat_totals = cat_totals.sort_values('Total_Spent', ascending=False)
print(cat_totals.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(cat_totals['Total_Spent'], labels=cat_totals.index,
            autopct='%1.1f%%', startangle=140)
axes[0].set_title('Category-wise Expense Distribution')

# Bar chart
cat_totals['Total_Spent'].plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Total Spending by Category')
axes[1].set_xlabel('Amount (₹)')

plt.tight_layout()
plt.show()

## 3. Monthly Trends

In [ ]:
df_exp['Month_Year'] = df_exp['Date'].dt.to_period('M').astype(str)
monthly = df_exp.groupby('Month_Year')['Amount'].sum().reset_index()

plt.figure(figsize=(14, 4))
plt.plot(monthly['Month_Year'], monthly['Amount'], marker='o', color='steelblue', linewidth=2)
plt.fill_between(monthly['Month_Year'], monthly['Amount'], alpha=0.2, color='steelblue')
plt.xticks(rotation=60, fontsize=8)
plt.title('Monthly Total Expense Trend')
plt.ylabel('Amount (₹)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Weekend vs Weekday Analysis

In [ ]:
wk_analysis = df_exp.groupby('Is_Weekend')['Amount'].agg(['sum','mean','count'])
wk_analysis.index = ['Weekday', 'Weekend']
print(wk_analysis)

wk_analysis['mean'].plot(kind='bar', color=['steelblue','coral'], rot=0)
plt.title('Average Transaction: Weekend vs Weekday')
plt.ylabel('Avg Amount (₹)')
plt.show()

## 5. Correlation & Heatmap

In [ ]:
pivot = df_exp.groupby(['Month_Year','Category'])['Amount'].sum().unstack(fill_value=0)
pivot_12 = pivot.tail(12)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_12.T, cmap='YlOrRd', annot=True, fmt='.0f',
            annot_kws={'size':6}, linewidths=0.5)
plt.title('Monthly Spending Heatmap (Last 12 Months)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## 6. Payment Method Analysis

In [ ]:
pm = df_exp.groupby('Payment_Method')['Amount'].agg(['sum','count'])
pm.columns = ['Total_Spent','Transactions']
pm = pm.sort_values('Total_Spent', ascending=False)
print(pm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pm['Total_Spent'].plot(kind='bar', ax=axes[0], color='steelblue', rot=30)
axes[0].set_title('Total Spent by Payment Method')
pm['Transactions'].plot(kind='bar', ax=axes[1], color='coral', rot=30)
axes[1].set_title('Number of Transactions')
plt.tight_layout()
plt.show()

## 7. Key Findings Summary

In [ ]:
print('='*50)
print('KEY FINDINGS')
print('='*50)
print(f'Total Transactions : {len(df_exp):,}')
print(f'Total Spent        : ₹{df_exp["Amount"].sum():,.0f}')
print(f'Average Transaction: ₹{df_exp["Amount"].mean():,.0f}')
print(f'Date Range         : {df_exp["Date"].min().date()} to {df_exp["Date"].max().date()}')
print(f'Top Category       : {df_exp.groupby("Category")["Amount"].sum().idxmax()}')
print(f'Top Payment Method : {df_exp.groupby("Payment_Method")["Amount"].sum().idxmax()}')
print(f'Weekend Premium    : {df_exp.groupby("Is_Weekend")["Amount"].mean()[1]/df_exp.groupby("Is_Weekend")["Amount"].mean()[0]*100-100:.1f}% higher spending on weekends')